# GEOG 499, Unit 1: Dashboard (Aaron Goodman)

## 📜 Usage Guide

Hello! Thanks for trying out my map and data dashboard. This dashboard serves as a data exploration tool for various environmental metrics in Lake Tahoe which are calculated on the fly for a provided parcel and ["buffer"](https://en.wikipedia.org/wiki/Buffer_analysis) distance. Calculated metrics include:
<br>
- number of transit stops within buffer distance 🔵
- length (feet, miles) of existing bike trails 🚲
- length (feet, miles) of proposed bike trails 🚲
- nearest fire station; distance (feet) to nearest fire station 🚒
- flood zone status (parcel in 100- or 500-year flood zone) 🌊
- soil types intersecting parcel 🌱
- recreation features (incl. trailheads) within buffer distance 🏕

The "Code" section has 3 subsections, which must be run in order (Google Colab will handle this with 'Runtime > Run all' or Ctrl+F9):
<br>
1. Package Install & Import
  - `pandas`, `geopandas`, `folium`, and `ipywidgets` will be installed
  - these 4 libraries will be imported along with certain functions from:
  <br>    `IPython.display`, `shapely.geometry`, and `xyzservices.providers`
2. Data Load
  - All 10 layers used by the dashboard will be imported from `agg_geog499_u1.gpkg`, which should be packaged with this notebook. Make sure the GeoPackage is added to your Colab runtime (in the 'Files' section of the left toolbar) and that its upload has completed.
3. Dashboard
  - The code used to run the dashboard itself is contained in one last, large block. Here's a brief technical rundown of how it works, followed by a more comprehensive guide to the UI:
  <br>    `Technical Rundown`
    - The dashboard has 2 input widgets: one for parcel selection via APN, and another for buffer distance in meters
    - The "Go!" button triggers the (re)drawing of the map and (re)calculation of metrics
      - Most of the logic of the dashboard is contained in the `on_button_click` function triggered by this button:
        - all output is cleared
        - input APN and buffer distance are taken, status is printed
        - parcel is selected by APN, buffer is created
        - metrics are calculated with `geopandas`; then printed
        - selected parcel and buffer are reprojected to EPSG:4326; added to map
        - map extent is adjusted according to buffer geometry
        - some metrics layers are reprojected to EPSG:4326; added to map
        - layer control is added and map is displayed
    - The "Show Example APNs" button prints real sampled APN values from the dataset to help start data exploration
    - Container widgets are used to structure all 3 types of output into a grid: status messages, the map, and metrics calculated from data
    
    📜 `UI Guide`
    - Type in the APN (Assessor's Parcel Number) for the parcel you're looking for. If you're not sure which parcel you're looking for, click the "Show Example APNs" button to see some real APNs in Lake Tahoe.
    - Type in your preferred buffer distance if you would like to search beyond the default 100m. Note that the map will take longer to load for searches which include a buffer of greater than 1000m (1km).
    - Press the "Go!" button.
      - The "Metrics" should load almost instantly. While your map loads, take a look at the data calculated for your selected parcel and buffer distance: Transit Stops (count within buffer), Bike Trails (total length within buffer, existing and proposed), Nearest Fire Station (and distance), Flood Zone Status, Soil Type, and Recreation Features (count/list within buffer distance).
      - The map will show the selected parcel in pink with a transparent blue buffer around it. The map extent will adjust according to the location of your parcel and the size of your buffer; depending on these factors, you may be able to see several other features on the map:
        - Transit stops appear as blue dots
        - Bike trails are drawn over the roads visible in the underlying basemap (dashed lines are proposed trails)
        - Fire stations appear with orange icons, except for the station closest to the selected parcel, which has a red icon
        - Recreation sites and trailheads appear with unique icons
      - Note that the map is interactive! All of the features listed above will, if clicked on, trigger pop-ups that tell you the name of the stop/trail/station/site.
      - You can click "Show Example APNs" repeatedly to see 10 new valid APNs if you want to keep exploring, but don't know parcel numbers!

## 📊 Data

### TRPA ArcGIS REST Services
- (1) parcels: [MapServer: `DataDownloader_Parcels`, Layer: `Parcels`](https://maps.trpa.org/server/rest/services/DataDownloader_Parcels/MapServer/0)
- (2) transitStops: [MapServer: `LTinfo_Climate_Reslience_Dashboard`, Layer: `Transit Stops`](https://maps.trpa.org/server/rest/services/LTinfo_Climate_Resilience_Dashboard/MapServer/27)
- (3) bikeTrailsExisting: [MapServer: `DataDownloader_Transportation`, Layer: `Bike Trails Existing`](https://maps.trpa.org/server/rest/services/DataDownloader_Transportation/MapServer/3)
- (4) bikeTrailsProposed: [MapServer: `DataDownloader_Transportation`, Layer: `Bike Trails Proposed`](https://maps.trpa.org/server/rest/services/DataDownloader_Transportation/MapServer/4)
- (5) fireStations: [MapServer: `Emergency_Services`, Layer: `Fire Station`](https://maps.trpa.org/server/rest/services/Emergency_Services/MapServer/3)
- (6) floodZone100: [MapServer: `DataDownloader_SoilsandHydro`, Layer: `100 Year Flood Zone`](https://maps.trpa.org/server/rest/services/DataDownloader_SoilsandHydro/MapServer/0)
- (7) floodZone500: [MapServer: `DataDownloader_SoilsandHydro`, Layer: `500 Year Flood Zone`](https://maps.trpa.org/server/rest/services/DataDownloader_SoilsandHydro/MapServer/27)
- (8) soilSurvey2003: [MapServer: `DataDownloader_SoilsandHydro`, Layer: `Soil Survey - NRCS 2003`](https://maps.trpa.org/server/rest/services/DataDownloader_SoilsandHydro/MapServer/8)
- (9) recSites: [MapServer: `DataDownloader_Recreation`, Layer: `Recreation Sites`](https://maps.trpa.org/server/rest/services/DataDownloader_Recreation/MapServer/2)
- (10) trailheads: [MapServer: `DataDownloader_Recreation`, Layer: `Trailheads`](https://maps.trpa.org/server/rest/services/DataDownloader_Recreation/MapServer/1)

Layers were loaded into ArcGIS Pro via ArcGIS Online, then copied into GeoPackage.


## ✅ Project Checklist

### Dashboard

#### Map
- `*.ipynb`: load TRPA parcels data, identify selected parcel by APN, create buffer (handling projection properly)
  - APN ID: `ipywidgets` input(APN, Buffer Radius [m])
- `folium` map:
  - display selected parcel in distinct color
  - buffer appears as a semi-transparent polygon
  - load 3+ additional layers for analysis component
  - automatically handle zoom & extent
- *"use callbacks to ensure that clicking on a button after entering APN or radius automatically updates the analysis and map"*

#### Analysis
- use `gpd` for spatial analysis
  - length property on trail geometries
  - handle null values properly
  - present results clearly
```{csv}
Name,    Dataset,    Metric
`transit stops`,  Tahoe Transit Stops,  `#/stops within buffer`
`bike trails`,  Active Transportation Plan:Existing/Proposed Bike Trails,   `total length (ft/mi) of existing/proposed trails`
`fire stations`,  Tahoe Fire Stations,  `distance to nearest F.D.: boundary and centroid`
`flood zones`,  FEMA Flood Zones,  `TRUE/FALSE: intersects 100- or 500-yr FEMA floodzone`
`soil types`,  Soil Survey 2003,  `list intersecting soil types (parcel; not buffer)`
`rec. sites & trailheads`,  Tahoe Recreation Sites,  `list rec. features (parks/trailheads) within buffer`
```

## 💻 Code

### Package Install & Import

In [1]:
! pip install pandas geopandas folium ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 19.0 MB/s eta 0:00:00


In [2]:
import pandas as pd
import geopandas as gpd
import folium
import xyzservices.providers as xyz
import ipywidgets as widgets
from IPython.display import display, clear_output
from shapely.geometry import Polygon, LineString, Point, MultiPolygon

### Data Load

In [3]:
# Add this GeoPackage to the 'Files' in your Colab runtime
gpkg_path = 'agg_geog499_u1.gpkg'

### ===== LAYERS =====
### all with EPSG:26910 NAD83 / UTM zone 10N

# ===============
# === parcels ===
# ===============
parcels = gpd.read_file(gpkg_path, layer='parcels')
# Timestamp fields are causing trouble with Folium
parcels = parcels.drop('created_date', axis=1)
parcels = parcels.drop('last_edited_date', axis=1)

# ====================
# === transitStops ===
# ====================
transitStops = gpd.read_file(gpkg_path, layer='transitStops')
# Timestamp
transitStops = transitStops.drop('created_date', axis=1)
transitStops = transitStops.drop('last_edited_date', axis=1)

# ===================
# === bikeTrails* ===
# ===================
## .Existing
bikeTrailsExisting = gpd.read_file(gpkg_path, layer='bikeTrailsExisting')
# Timestamp
bikeTrailsExisting = bikeTrailsExisting.drop('created_date', axis=1)
bikeTrailsExisting = bikeTrailsExisting.drop('last_edited_date', axis=1)
## .Proposed
bikeTrailsProposed = gpd.read_file(gpkg_path, layer='bikeTrailsProposed')
# Timestamp
bikeTrailsProposed = bikeTrailsProposed.drop('created_date', axis=1)
bikeTrailsProposed = bikeTrailsProposed.drop('last_edited_date', axis=1)

# ====================
# === fireStations ===
# ====================
fireStations = gpd.read_file(gpkg_path, layer='fireStations')
# Timestamp
fireStations = fireStations.drop('created_date', axis=1)
fireStations = fireStations.drop('last_edited_date', axis=1)

# ==================
# === floodZone* ===
# ==================
floodZone100 = gpd.read_file(gpkg_path, layer='floodZone100')
floodZone500 = gpd.read_file(gpkg_path, layer='floodZone500')

# ======================
# === soilSurvey2003 ===
# ======================
soilSurvey2003 = gpd.read_file(gpkg_path, layer='soilSurvey2003')
# Timestamp
soilSurvey2003 = soilSurvey2003.drop('created_date', axis=1)
soilSurvey2003 = soilSurvey2003.drop('last_edited_date', axis=1)

# ================
# === recSites ===
# ================
recSites = gpd.read_file(gpkg_path, layer='recSites')
# Timestamp
recSites = recSites.drop('created_date', axis=1)
recSites = recSites.drop('last_edited_date', axis=1)

# ==================
# === trailheads ===
# ==================
trailheads = gpd.read_file(gpkg_path, layer='trailheads')
# Timestamp
trailheads = trailheads.drop('created_date', axis=1)
trailheads = trailheads.drop('last_edited_date', axis=1)

/usr/local/lib/python3.11/dist-packages/pyogrio/raw.py:198: RuntimeWarning: Non-conformant content for record 1 in column last_edited_date, 2024-12-28T01:04:27.0Z, successfully parsed
  return ogr_read(
/usr/local/lib/python3.11/dist-packages/pyogrio/raw.py:198: RuntimeWarning: Non-conformant content for record 2 in column last_edited_date, 2024-02-16T17:08:21.0Z, successfully parsed
  return ogr_read(
/usr/local/lib/python3.11/dist-packages/pyogrio/raw.py:198: RuntimeWarning: Non-conformant content for record 1 in column created_date, 2024-01-25T04:41:45.0Z, successfully parsed
  return ogr_read(
/usr/local/lib/python3.11/dist-packages/pyogrio/raw.py:198: RuntimeWarning: Non-conformant content for record 1 in column created_date, 2024-01-25T04:36:33.0Z, successfully parsed
  return ogr_read(


### Dashboard

In [6]:
# Widgets
apn_box = widgets.Text(
    value='', # default APN blank
    placeholder="Enter APN",
    description="APN:",
    disabled=False)
buffer_box = widgets.FloatText(
    value=100.0, # default 100 meters
    description="Buffer (m):",
    disabled=False)
go_button = widgets.Button(
    description="Go!",
    disabled=False,
    button_style="primary",
    tooltip="Click to update map")
example_button = widgets.Button(
    description="Show Example APNs",
    button_style="info",
    tooltip="Click to see example valid APNs")
## status_container
status_container = widgets.VBox([
    widgets.HTML(value="<h3>Dashboard Status</h3>"),
    widgets.Output(
        layout=widgets.Layout(
            width='400px',
            height='200px',
            border='1px solid #888',
            overflow='auto',
            flex='1'
        )
    )
], layout=widgets.Layout(
    width='400px',
    height='250px',
    display='flex',
    flex_flow='column',
    align_items='stretch'
))
## map_container
map_container = widgets.VBox([
    widgets.HTML(value="<h3>Map</h3>"),
    widgets.Output(
        layout=widgets.Layout(
            width='1200px',
            height='600px',
            border='1px solid #888',
            overflow='hidden'
        )
    )
], layout=widgets.Layout(
    width='1200px',
    height='650px',
    display='flex',
    flex_flow='column',
    align_items='stretch'
))
## metrics_container
metrics_container = widgets.VBox([
    widgets.HTML(value="<h3>Environmental Metrics</h3>"),
    widgets.Output(
        layout=widgets.Layout(
            width='500px',
            height='400px',
            border='1px solid #888',
            overflow='auto',
            flex='1'
        )
    )
], layout=widgets.Layout(
    width='500px',
    height='450px',
    display='flex',
    flex_flow='column',
    align_items='stretch'
))
## Access the output widgets directly from the containers
status_output = status_container.children[1]
map_output = map_container.children[1]
metrics_output = metrics_container.children[1]
# Define metric_results globally
metric_results = {
    'Metric': [],
    'Value': []
}
## Display metrics
with metrics_output:
    # print each metric on its own line
    for item in metric_results:
        print(item)

# Grid
grid = widgets.GridspecLayout(2, 6, height='750px')
## Row 1: Input widgets
grid[0, 0:2] = apn_box
grid[0, 2:4] = buffer_box
grid[0, 4] = go_button
grid[0, 5] = example_button
## Row 2: Status, Map, and Metrics
grid[1, 1] = status_container
grid[1, 2:3] = map_container
grid[1, 3:5] = metrics_container

# Map
## Example_button for APN examples
def show_example_apns():
    """Display a few example valid APNs"""
    print("Example valid APNs:")
    print(parcels['APN'].sample(10).values)
def on_example_button_click(b):
    show_example_apns()
example_button.on_click(on_example_button_click)

## Main dashboard logic
def on_button_click(b):
    # Clear output areas
    status_output.clear_output()
    metrics_output.clear_output()
    map_output.clear_output()

    # Initialize metric_results as a dictionary
    metric_results = {
        'Metric': [],
        'Value': []
    }

    # Get input values
    input_apn = apn_box.value
    buffer_distance = buffer_box.value

    # Display status messages
    with status_output:
        print(f"Selected APN: {input_apn}")
        print(f"Selected Buffer: {buffer_distance} meters")

        # Validate APN
        if input_apn not in parcels['APN'].values:
            print(f"Error: APN '{input_apn}' not found in parcels dataset.")
            print("Please enter a valid APN and try again.")
            return

        # Debug info
        print(f"Found {len(parcels[parcels['APN'] == input_apn])} matching parcels")
        print("The map will take some time to load... ⛾")

    # Find selected parcel by APN
    selected_parcel = parcels[parcels['APN'] == input_apn].copy()

    # Make sure we're working with a single geometry
    if len(selected_parcel) > 1:
        with status_output:
            print(f"Warning: Multiple parcels found with APN {input_apn}. Using the first one.")
        selected_parcel = selected_parcel.iloc[[0]]

    # Get the single geometry object
    parcel_geom = selected_parcel.geometry.iloc[0]

    # Create buffer of specified distance
    buffer_geom = parcel_geom.buffer(buffer_distance)

    # Create gdf for the buffer
    buffer_gdf = gpd.GeoDataFrame(
        {'APN': [selected_parcel['APN'].values[0]], 'buffer_distance': [buffer_distance]},
        geometry=[buffer_geom],
        crs=selected_parcel.crs
    )

    # =============================================
    # METRIC CALCULATIONS START HERE
    # =============================================

    # 1. Transit Stops Count
    try:
        transit_stops_in_buffer = transitStops[transitStops.intersects(buffer_geom)]
        transit_stops_count = len(transit_stops_in_buffer)

        metric_results['Metric'].append('Transit Stops')
        metric_results['Value'].append(f"{transit_stops_count} within buffer")
    except Exception as e:
        metric_results['Metric'].append('Transit Stops')
        metric_results['Value'].append(f"Error: {str(e)}")

    # 2. Bike Trails Length
    try:
        # Existing bike trails
        existing_trails_in_buffer = bikeTrailsExisting[bikeTrailsExisting.intersects(buffer_geom)]
        if not existing_trails_in_buffer.empty:
            existing_trails_clipped = gpd.clip(existing_trails_in_buffer, buffer_gdf)
            existing_trails_length_feet = existing_trails_clipped.length.sum() * 3.28084  # Convert meters to feet
            existing_trails_length_miles = existing_trails_length_feet / 5280
        else:
            existing_trails_length_feet = 0
            existing_trails_length_miles = 0

        # Proposed bike trails
        proposed_trails_in_buffer = bikeTrailsProposed[bikeTrailsProposed.intersects(buffer_geom)]
        if not proposed_trails_in_buffer.empty:
            proposed_trails_clipped = gpd.clip(proposed_trails_in_buffer, buffer_gdf)
            proposed_trails_length_feet = proposed_trails_clipped.length.sum() * 3.28084  # Convert meters to feet
            proposed_trails_length_miles = proposed_trails_length_feet / 5280
        else:
            proposed_trails_length_feet = 0
            proposed_trails_length_miles = 0

        metric_results['Metric'].append('Existing Bike Trails')
        metric_results['Value'].append(f"{existing_trails_length_feet:.2f} feet ({existing_trails_length_miles:.2f} miles)")

        metric_results['Metric'].append('Proposed Bike Trails')
        metric_results['Value'].append(f"{proposed_trails_length_feet:.2f} feet ({proposed_trails_length_miles:.2f} miles)")
    except Exception as e:
        metric_results['Metric'].append('Bike Trails')
        metric_results['Value'].append(f"Error: {str(e)}")

    # 3. Distance to Nearest Fire Station
    try:
        # Calculate centroid of the selected parcel
        parcel_centroid = parcel_geom.centroid

        if not fireStations.empty:
            # Calculate distances from centroid to all fire stations
            distances = fireStations.geometry.apply(lambda x: parcel_centroid.distance(x))

            # Find the minimum distance
            min_distance = distances.min()
            nearest_station_idx = distances.idxmin()

            # Get the name of the nearest station if available
            if 'NAME' in fireStations.columns:
                nearest_station_name = fireStations.iloc[nearest_station_idx]['NAME']
            elif 'STATION_NA' in fireStations.columns:
                nearest_station_name = fireStations.iloc[nearest_station_idx]['STATION_NA']
            else:
                nearest_station_name = f"Station #{nearest_station_idx}"

            # Convert to feet
            min_distance_feet = min_distance * 3.28084

            metric_results['Metric'].append('Nearest Fire Station')
            metric_results['Value'].append(f"{nearest_station_name} ({min_distance_feet:.2f} feet)")
        else:
            metric_results['Metric'].append('Nearest Fire Station')
            metric_results['Value'].append("No fire stations found in the dataset.")
    except Exception as e:
        metric_results['Metric'].append('Nearest Fire Station')
        metric_results['Value'].append(f"Error: {str(e)}")

    # 4. Flood Zone Intersection
    try:
        # Check if parcel intersects with 100-year flood zone
        in_100yr_flood = False
        if not floodZone100.empty:
            in_100yr_flood = any(floodZone100.intersects(parcel_geom))

        # Check if parcel intersects with 500-year flood zone
        in_500yr_flood = False
        if not floodZone500.empty:
            in_500yr_flood = any(floodZone500.intersects(parcel_geom))

        # Status message
        if in_100yr_flood and in_500yr_flood:
            flood_status = "In both 100-year and 500-year flood zones"
        elif in_100yr_flood:
            flood_status = "In 100-year flood zone"
        elif in_500yr_flood:
            flood_status = "In 500-year flood zone"
        else:
            flood_status = "Not in 100-year or 500-year flood zones"

        metric_results['Metric'].append('Flood Zone Status')
        metric_results['Value'].append(flood_status)
    except Exception as e:
        metric_results['Metric'].append('Flood Zone Status')
        metric_results['Value'].append(f"Error: {str(e)}")

    # 5. Soil Types Intersection
    try:
        # Find soil types that intersect with the parcel
        intersecting_soils = soilSurvey2003[soilSurvey2003.intersects(parcel_geom)]

        if not intersecting_soils.empty and 'MUNAME' in intersecting_soils.columns:
            soil_types = intersecting_soils['MUNAME'].unique().tolist()

            if soil_types:
                metric_results['Metric'].append('Soil Types')
                metric_results['Value'].append(', '.join(map(str, soil_types)))
            else:
                metric_results['Metric'].append('Soil Types')
                metric_results['Value'].append("No soil types found intersecting the parcel.")
        else:
            metric_results['Metric'].append('Soil Types')
            metric_results['Value'].append("No soil type information available.")
    except Exception as e:
        metric_results['Metric'].append('Soil Types')
        metric_results['Value'].append(f"Error: {str(e)}")

    # 6. Recreation Features (Sites and Trailheads)
    try:
        # Find recreation sites within buffer
        rec_sites_in_buffer = recSites[recSites.intersects(buffer_geom)]

        # Find trailheads within buffer
        trailheads_in_buffer = trailheads[trailheads.intersects(buffer_geom)]

        # Count total features
        total_rec_features = len(rec_sites_in_buffer) + len(trailheads_in_buffer)

        # Get names of recreation sites
        rec_site_names = []
        if not rec_sites_in_buffer.empty and 'RECREATION_NAME' in rec_sites_in_buffer.columns:
            rec_site_names = rec_sites_in_buffer['RECREATION_NAME'].tolist()

        # Get names of trailheads
        trailhead_names = []
        if not trailheads_in_buffer.empty and 'RECREATION_NAME' in trailheads_in_buffer.columns:
            trailhead_names = trailheads_in_buffer['RECREATION_NAME'].tolist()

        # Combine all feature names
        all_feature_names = rec_site_names + trailhead_names

        # Add to metrics
        metric_results['Metric'].append('Recreation Features')
        if total_rec_features > 0:
            if all_feature_names:
                # If the list is very long, truncate it
                if len(all_feature_names) > 5:
                    feature_list = ", ".join(all_feature_names[:5]) + f", and {len(all_feature_names) - 5} more"
                else:
                    feature_list = ", ".join(all_feature_names)
                metric_results['Value'].append(f"{total_rec_features} within buffer: {feature_list}")
            else:
                metric_results['Value'].append(f"{total_rec_features} unnamed features within buffer")
        else:
            metric_results['Value'].append("No recreation features within buffer")
    except Exception as e:
        metric_results['Metric'].append('Recreation Features')
        metric_results['Value'].append(f"Error: {str(e)}")

    # Display metrics as a table
    with metrics_output:
        metrics_df = pd.DataFrame(metric_results)
        display(metrics_df.set_index('Metric').style.set_properties(**{
            'text-align': 'left',
            'white-space': 'pre-wrap',
            'font-size': '12px',
            'padding': '5px'
        }))

    # =============================================
    # METRIC CALCULATIONS END HERE
    # =============================================

    # Reproject Buffer to WGS84 for map display
    buffer_gdf = buffer_gdf.to_crs(epsg=4326)

    # Remove the centroid column from selected_parcel before reprojecting
    if 'centroid' in selected_parcel.columns:
        selected_parcel = selected_parcel.drop(columns=['centroid'])
    selected_parcel_4326 = selected_parcel.to_crs(epsg=4326)

    # Get the bounds of the buffer for map centering
    buffer_bounds = buffer_gdf.total_bounds
    map_center = [(buffer_bounds[1] + buffer_bounds[3])/2,
                (buffer_bounds[0] + buffer_bounds[2])/2]

    # Create a new map with controlled size
    new_map = folium.Map(
        location=map_center,
        tiles=xyz.OpenTopoMap,
        width='100%',
        height='100%'
    )

    # Add the selected parcel
    folium.GeoJson(
        selected_parcel_4326.__geo_interface__,
        name='Selected Parcel',
        style_function=lambda x: {'fillColor': '#ff0000', 'color': '#000000',
                                'fillOpacity': 0.5, 'weight': 1},
        popup=folium.GeoJsonPopup(
            fields=['APN'],
            aliases=['Selected APN:'],
            localize=True,
            labels=True,
            style="font-weight: bold;"
        )
    ).add_to(new_map)

    # Add the buffer
    folium.GeoJson(
        buffer_gdf.__geo_interface__,
        name='Buffer Zone',
        style_function=lambda x: {'fillColor': '#0000ff', 'color': '#0000ff',
                                'fillOpacity': 0.2, 'weight': 1},
        control=True,
        show=True,
        interactive=False
    ).add_to(new_map)

    # =============================================
    # LAYERS FROM METRIC CALCS START HERE
    # =============================================
    # Reproject all layers to WGS84 for map display
    transitStops_4326 = transitStops.to_crs(epsg=4326)
    fireStations_4326 = fireStations.to_crs(epsg=4326)
    bikeTrailsExisting_4326 = bikeTrailsExisting.to_crs(epsg=4326)
    bikeTrailsProposed_4326 = bikeTrailsProposed.to_crs(epsg=4326)
    recSites_4326 = recSites.to_crs(epsg=4326)
    trailheads_4326 = trailheads.to_crs(epsg=4326)
    #### Drawn in order different from calculation
    # 2. Fire Stations Layer with special styling for nearest
    fire_stations_group = folium.FeatureGroup(name='Fire Stations')
    # Find nearest fire station (already calculated in metrics)
    if not fireStations.empty:
        # Calculate distances from centroid to all fire stations
        parcel_centroid = parcel_geom.centroid
        distances = fireStations.geometry.apply(lambda x: parcel_centroid.distance(x) if x is not None else float('inf'))
        nearest_station_idx = distances.idxmin()
        for idx, row in fireStations_4326.iterrows():
            # Skip if geometry is None
            if row.geometry is None:
                continue
            # Check geometry
            if hasattr(row.geometry, 'x') and hasattr(row.geometry, 'y'):
                # Determine if this is the nearest station
                is_nearest = (idx == nearest_station_idx)
                # Get station name
                station_name = row.get('NAME', f"Station #{idx}")
                # Style based on whether it's the nearest
                if is_nearest:
                    icon = folium.Icon(color='red', icon='fire-extinguisher', prefix='fa')
                    popup_content = f"<b>NEAREST STATION</b><br>{station_name}"
                else:
                    icon = folium.Icon(color='orange', icon='fire-extinguisher', prefix='fa')
                    popup_content = f"Fire Station: {station_name}"

                folium.Marker(
                    location=[row.geometry.y, row.geometry.x],
                    icon=icon,
                    popup=popup_content
                ).add_to(fire_stations_group)
    fire_stations_group.add_to(new_map)

    # 3. Existing Bike Trails Layer
    bike_existing_group = folium.FeatureGroup(name='Existing Bike Trails')
    #  Color mapping
    existing_class_colors = {
        '1': '#228B22',
        '2': '#1E90FF',
        '3': '#FFD700',
        'PED': '#9932CC',
        'default': '#808080'
    }
    for idx, row in bikeTrailsExisting_4326.iterrows():
        # Skip if geometry is None
        if row.geometry is None:
            continue
        # Get the class value, defaulting to 'default' if not found
        trail_class = str(row.get('CLASS', 'default'))
        color = existing_class_colors.get(trail_class, existing_class_colors['default'])
        # Create popup with trail information
        popup_content = f"Bike Trail: {row.get('NAME', 'Unnamed')}<br>Class: {trail_class}"
        # Add the line to the map
        folium.GeoJson(
            row.geometry.__geo_interface__,
            style_function=lambda x, color=color: {
                'color': color,
                'weight': 3,
                'opacity': 0.8
            },
            popup=folium.Popup(popup_content)
        ).add_to(bike_existing_group)
    bike_existing_group.add_to(new_map)

    # 4. Proposed Bike Trails Layer (dashed lines)
    bike_proposed_group = folium.FeatureGroup(name='Proposed Bike Trails')
    # Color mapping
    proposed_class_colors = {
        '0': '#A9A9A9',
        '1': '#228B22',
        '2': '#1E90FF',
        '3': '#FFD700',
        '5': '#FF4500',
        'default': '#808080'
    }
    for idx, row in bikeTrailsProposed_4326.iterrows():
        # Skip if geometry is None
        if row.geometry is None:
            continue
        # Get the class value, defaulting to 'default' if not found
        trail_class = str(row.get('CLASS', 'default'))
        color = proposed_class_colors.get(trail_class, proposed_class_colors['default'])
        # Create popup with trail information
        popup_content = f"Proposed Bike Trail: {row.get('NAME', 'Unnamed')}<br>Class: {trail_class}"
        # Add the line to the map with dashed style
        folium.GeoJson(
            row.geometry.__geo_interface__,
            style_function=lambda x, color=color: {
                'color': color,
                'weight': 3,
                'opacity': 0.8,
                'dashArray': '5, 5'
            },
            popup=folium.Popup(popup_content)
        ).add_to(bike_proposed_group)
    bike_proposed_group.add_to(new_map)

    # 5. Recreation Sites Layer
    rec_sites_group = folium.FeatureGroup(name='Recreation Sites')
    for idx, row in recSites_4326.iterrows():
        # Skip if geometry is None
        if row.geometry is None:
            continue
        # Check geometry
        if hasattr(row.geometry, 'x') and hasattr(row.geometry, 'y'):
            # Get site name
            site_name = row.get('RECREATION_NAME', 'Unnamed Recreation Site')
            folium.Marker(
                location=[row.geometry.y, row.geometry.x],
                icon=folium.Icon(color='green', icon='tree', prefix='fa'),
                popup=f"Recreation Site: {site_name}"
            ).add_to(rec_sites_group)
    rec_sites_group.add_to(new_map)

    # 6. Trailheads Layer
    trailheads_group = folium.FeatureGroup(name='Trailheads')
    for idx, row in trailheads_4326.iterrows():
        # Skip if geometry is None
        if row.geometry is None:
            continue
        # Check geometry
        if hasattr(row.geometry, 'x') and hasattr(row.geometry, 'y'):
            # Get trailhead name
            trailhead_name = row.get('RECREATION_NAME', 'Unnamed Trailhead')

            folium.Marker(
                location=[row.geometry.y, row.geometry.x],
                icon=folium.Icon(color='purple', icon='hiking', prefix='fa'),
                popup=f"Trailhead: {trailhead_name}"
            ).add_to(trailheads_group)
    trailheads_group.add_to(new_map)

    # 1. Transit Stops Layer
    transit_stops_group = folium.FeatureGroup(name='Transit Stops')
    for idx, row in transitStops_4326.iterrows():
        # Skip if geometry is None
        if row.geometry is None:
            continue
        # Check geometry
        if hasattr(row.geometry, 'x') and hasattr(row.geometry, 'y'):
            popup_content = f"Transit Stop: {row.get('STOP_NAME', 'Unnamed')}"
            folium.CircleMarker(
                location=[row.geometry.y, row.geometry.x],
                radius=3,
                color='blue',
                fill=True,
                fill_color='blue',
                fill_opacity=0.7,
                popup=popup_content
            ).add_to(transit_stops_group)
    transit_stops_group.add_to(new_map)
    # =============================================
    # LAYERS FROM METRIC CALCS END HERE
    # =============================================

    # Add a layer control
    folium.LayerControl().add_to(new_map)

    # Fit the map to the buffer bounds
    new_map.fit_bounds([
        [buffer_bounds[1], buffer_bounds[0]],  # SW corner
        [buffer_bounds[3], buffer_bounds[2]]   # NE corner
    ])

    # Display the map in the map output area
    with map_output:
        display(new_map)

# Button click handler
go_button.on_click(on_button_click)

# Initial display
clear_output(wait=True)
display(grid)
with status_output:
    print("Enter an APN and buffer distance, then click 'Go!' to update the map.")
with metrics_output:
    print("Select a parcel to view environmental metrics analysis.")

GridspecLayout(children=(Text(value='', description='APN:', layout=Layout(grid_area='widget001'), placeholder=…

Example valid APNs:
['098-350-001' '016-142-013' '122-213-06' '117-080-052' '032-362-008'
 '122-052-05' '016-523-006' '097-193-005' '036-423-001' '132-066-41']
Example valid APNs:
['034-084-007' '094-420-008' '116-100-038' '1318-10-312-020' '097-083-005'
 '085-113-014' '1318-23-611-001' '125-174-20' '034-623-005' '034-153-010']
Example valid APNs:
['128-033-08' '027-010-024' '025-911-058' '022-311-023' '097-242-025'
 '128-270-00' '116-200-025' '015-215-003' '126-171-06' '1319-19-802-010']
